# **Install and Import Libraries**

> ##### **Make sure the secrets.env file is in the config folder. An example for secrets.env can be found in config/secrets_example.env file**


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from dotenv import load_dotenv

# load config
load_dotenv("../config/configOld.env")

# load secrets
load_dotenv("../config/secrets.env")

In [ ]:
import os

# Override the standard variables with the upcoming variables
# This should probably be done in a better way in the future
os.environ["SCRAPING_START_URL"] = os.getenv("UPCOMING_SCRAPING_START_URL")
os.environ["PROTOCOLS_PATH"] = os.getenv("UPCOMING_PROTOCOLS_PATH")
os.environ["SCRAPED_DATA_FILE_PATH"] = os.getenv("UPCOMING_SCRAPED_DATA_FILE_PATH")

In [ ]:
from data_pipeline import *
import chatbot.llm_kg_retrieval as llm_kg_retrieval
import time

# **1. Scrape Website**
> Takes approximately 12 minutes to run.

> One can possibly use asyncronous functions to speed up this process.

In [ ]:
scrape_website()

# **2. Download all meeting documents from the scraped links**

> One can possibly use asyncronous functions to speed up this process.

In [ ]:
download_documents(overwrite=False)

# **3. Extract HTML and text from PDFs**

In [ ]:
# only converts pdf and docx files so it might be less than the downloaded files
# depth=3 only extracts agenda and protocols, depth=5 includes attachments
convert_files(depth=3, output_type="xhtml", overwrite=True, add_ids_to_tags=True)
convert_files(output_type="text", overwrite=True)

# **4. Extract Meeting Metadata**

In [ ]:
# Pass type=None to get all documents, so we just extract the unique meetings
full_df = get_documents_dataframe(type=None)

In [ ]:
# Printing for debugging
print("Columns in full_df:", full_df.columns.tolist())
print("\nNumber of rows:", len(full_df))
print("\nFirst few rows:")
print(full_df[['title', 'filepath', 'web_html_link']].head(10))

In [ ]:
# Generate the stub metadata files based purely on directory structure and scraper data
generate_upcoming_metadata(full_df)

# **5. Extract Meeting Agenda**

In [ ]:
agenda_df = get_documents_dataframe(type="agenda")
#agenda_df = agenda_df[agenda_df["body"] == "Kommunstyrelsen"] # filter for specific body for demonstration purposes - for faster completion

In [ ]:
# Printing for bugtesting
print("Columns in agenda_df:", agenda_df.columns.tolist())
print("\nNumber of rows:", len(agenda_df))
print("\nFirst few rows:")
print(agenda_df[['title', 'filepath', 'web_html_link']].head(10))

In [ ]:
# asynchronously extract meeting agenda (taking into account openai rate limits; limit defined in config file)
# await extract_meeting_data(df=agenda_df, type=type)

In [ ]:
# Batch extract meeting agenda for body
agenda_batch_id, references_batch_id = extract_meeting_data_batch(df=agenda_df, type="agenda",overwrite_data=True)
time.sleep(3)

In [ ]:
# If code has to be stopped while still waiting for batch to complete, uncomment and rerun from this node to pick back up
"""
# Check if there's a saved batch ID
batch_id_path = os.getenv("AGENDA_BATCH_INPUT_ID_SAVE_PATH")
if os.path.exists(batch_id_path):
    with open(batch_id_path, "r") as f:
        agenda_batch_id = f.read().strip()
    print(f"Found existing batch ID: {agenda_batch_id}")
    # Then you can directly use check_batch_status(agenda_batch_id)

references_batch_id = None
"""

In [ ]:
# Check batch status
print("Status for batch agenda extraction:")
agenda_output_id = None
while agenda_output_id is None:
    agenda_output_id = check_batch_status(agenda_batch_id)
    time.sleep(1)
agenda_output_jsonl = retrieve_batch_output(agenda_output_id)

# Only check references if references_batch_id is not None (i.e., there are documents with web_html_link)
if references_batch_id is not None:
    print("Status for batch references extraction:")
    references_output_id = None
    while references_output_id is None:
        references_output_id = check_batch_status(references_batch_id)
        time.sleep(1)
    references_output_jsonl = retrieve_batch_output(references_output_id)
else:
    references_output_jsonl = None
    print("No documents with web_html_link found for references extraction.")

In [ ]:
# Save batch results
await save_agenda_llm_batch_results(agenda_output_jsonl, agenda_df, references_jsonl=references_output_jsonl)

# Create html to preview the extracted agenda data, saved in notebooks folder
create_agenda_html(agenda_df)

# **6. Export JSON**

In [ ]:
construct_aggregate_json(construct_from="llm") # construct_from = "llm" or "manual"

# **7. Create a Knowledge Graph from JSON**

In [ ]:
# Caution: Set wipe_database=False to keep the existing old meetings and append upcoming ones to it
create_knowledge_graph(construct_from = "llm", wipe_database=False, is_upcoming=True) # construct_from = "llm" or "manual"

By default it will construct the knowledge graph from LLM extracted data. If you want to construct it from manually created JSON data, then add the data manually as follows:

1. Manually create JSON files with extracted data inside respective folders in `data/protocols` folder and name it `manual_meeting_metadata.json` or `manual_meeting_agenda.json` depending on the document type. Folder structure is `<body>`/`<meeting_date>`/`<document>`. Put the JSON inside the `<document>` folder.

2. Execute the `construct_aggregate_json(construct_from="manual")` function. This will fail if the created JSON does not follow the schema defined in `data/schema/schema.json`

3. Execute `create_knowledge_graph(constuct_from = "manual")` function.